# HyperSIGMA few-shot evaluation

Runs prototype-based few-shot evaluation against a frozen HyperSIGMA
dual-branch encoder (SpatViT + SpecViT + SEM). Optionally loads an
*adapted* checkpoint produced by `scripts/adapt_hypersigma_houston.py`;
set `adapted_checkpoint` to `"none"` for the unadapted ablation.

Everything for this run is written to
`experiments/<experiment_name>/evaluations/<eval_name>/`.

In [ ]:
# === Parameters ===
# The HyperSIGMA umbrella experiment to log under.
experiment_name = "hypersigma_baseline"

# Short eval-run identifier (becomes the eval subdir).
eval_name = "houston_15way_5shot_adapted_k3_fused"

# Path to the adapted checkpoint, or the string "none" for the unadapted ablation.
adapted_checkpoint = "checkpoints/hypersigma_adapted/houston_k3/checkpoint.pth"

# Evaluation hyperparameters. Anything you omit falls back to defaults
# in scripts.evaluate_hypersigma_cosine._DEFAULT_ARGS.
eval_params = {
    "dataset": "houston",
    "n_way": 15,
    "k_shot": 5,
    "k_query": 30,
    "num_episodes": 600,
    "split": "test",
    "seed": 42,
    "mode": "fused",          # 'fused' | 'spat_pool' | 'spec_pool'
    "spat_patch_k": 3,         # 3 (k=3, headline) | 1 (k=1 ablation)
    "pca_spat_path": "checkpoints/hypersigma/pca_houston_3band.pkl",
    "spat_ckpt": "checkpoints/hypersigma/spat-vit-base.pth",
    "spec_ckpt": "checkpoints/hypersigma/spec-vit-base.pth",
    "distance_metric": "cosine",
    "temperature": 10.0,
    "prototype_mode": "mean_features",
    "num_example_episodes": 3,
    "max_tsne_samples": 0,
    "no_plots": False,
}

overwrite = False

In [ ]:
import os, sys
from pathlib import Path
REPO = Path.cwd()
while not (REPO / "lib" / "experiments.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("Could not locate repo root containing lib/experiments.py")
    REPO = REPO.parent
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)

## Optional: fit the spatial PCA (run once per dataset)

In [ ]:
# Skip this cell if checkpoints/hypersigma/pca_<dataset>_3band.pkl already exists.
from pathlib import Path
pca_path = Path(eval_params["pca_spat_path"])
if not pca_path.exists():
    from data.datasets import HoustonPatchedDataset, TrentoPatchedDataset, MUUFLPatchedDataset
    from models.hypersigma.preprocessing import DATASET_PCA_CONFIG, fit_dataset_pca
    DS = {"houston": HoustonPatchedDataset, "trento": TrentoPatchedDataset, "muufl": MUUFLPatchedDataset}
    ds_name = eval_params["dataset"]
    dataset = DS[ds_name](data_root="./data/raw", patch_size=11, split="all", normalize=True)
    fit_dataset_pca(
        dataset=dataset,
        n_components=DATASET_PCA_CONFIG[ds_name]["spat_components"],
        save_path=str(pca_path),
    )
else:
    print(f"PCA already present at {pca_path}")

## Optional: run Level-2 MAE adaptation

Skip this cell when re-evaluating an already-adapted checkpoint or running
the unadapted ablation. The full 3000-epoch run takes a while; lower
`pretrain.epochs` in the config for smoke-testing.

In [ ]:
run_adaptation = False  # set True to run the MAE adaptation
if run_adaptation:
    from scripts.adapt_hypersigma_houston import run_adapt
    from utils.io import load_config
    cfg = load_config("configs/pretrain/hypersigma_houston_adapt.yaml")
    history = run_adapt(
        cfg,
        checkpoint_dir=cfg["paths"]["checkpoint_dir"],
        log_dir=cfg["paths"]["log_dir"],
    )
    print(history)

## Bootstrap the umbrella experiment dir (only the first time)

In [ ]:
from lib.experiments import ExperimentLogger
logger_ = ExperimentLogger(repo_root=REPO)
try:
    logger_.get_experiment(experiment_name)
    print(f"Experiment '{experiment_name}' already exists")
except FileNotFoundError:
    logger_.start_pretrain(
        name=experiment_name,
        description=(
            "HyperSIGMA baseline: SpatViT_fusion_patch + SpecViT_fusion + four-stage SEM, "
            "Houston Level-2 MAE adaptation on the small randomly-init components."
        ),
        config={"meta": "hypersigma baseline; pretraining is upstream + Houston adaptation"},
    )
    print(f"Created experiment '{experiment_name}'")

## Run evaluation

In [ ]:
from lib.eval_runner import run_hypersigma_evaluation

ev = run_hypersigma_evaluation(
    experiment_name=experiment_name,
    eval_name=eval_name,
    adapted_checkpoint=adapted_checkpoint,
    eval_params=eval_params,
    overwrite=overwrite,
)
print("Eval dir:", ev.root)

## Headline metrics (primary distance) and side-by-side cosine/euclidean

In [ ]:
import json
print("Summary (primary metric):")
print(json.dumps(ev.metadata.get("summary", {}), indent=2, default=str))
results = json.loads(ev.results_path.read_text())
for metric in ("cosine", "euclidean"):
    block = results.get(metric, {})
    print(f"\n{metric.upper():10s}  "
          f"OA={block.get('OA', {}).get('mean', float('nan')):.2f} +/- {block.get('OA', {}).get('ci_95', 0):.2f}  "
          f"AA={block.get('AA', {}).get('mean', float('nan')):.2f} +/- {block.get('AA', {}).get('ci_95', 0):.2f}  "
          f"Kappa={block.get('Kappa', {}).get('mean', float('nan')):.2f} +/- {block.get('Kappa', {}).get('ci_95', 0):.2f}")